Notebook 04 - UNSW-NB15 CNN Evaluation is completed locally.
Notebook 04 evaluated the baseline CNN model on the UNSW-NB15 dataset using TensorFlow/Keras. The model achieved 92.23% accuracy, 96.10% precision, 92.33% recall, and 94.18% F1-score, with a model size of approximately 0.536 MB. These results provide a second baseline for comparison with later lightweight optimization techniques such as pruning, quantization, and knowledge distillation.

In [13]:
import os

print(os.path.exists("../models/unsw_cnn_baseline_model.h5"))
print(os.path.exists("../models/unsw_scaler.pkl"))
print(os.path.exists("../results/unsw_cnn_baseline_results.csv"))
print(os.path.exists("../results/unsw_cnn_confusion_matrix.csv"))

True
True
True
True


In [12]:
model.save(MODEL_PATH)

model_size_mb = MODEL_PATH.stat().st_size / (1024 * 1024)

results_df = pd.DataFrame([
    {
        "Dataset": "UNSW-NB15",
        "Model": "Baseline CNN",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Training Time Seconds": training_time,
        "Inference Time Seconds": inference_time,
        "Model Size MB": model_size_mb
    }
])

results_df.to_csv(RESULTS_PATH, index=False)

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
)

cm_df.to_csv(CONFUSION_MATRIX_PATH)

print("Model saved to:", MODEL_PATH)
print("Results saved to:", RESULTS_PATH)
print("Confusion matrix saved to:", CONFUSION_MATRIX_PATH)

print("\nResults:")
print(results_df)

print("\nConfusion Matrix:")
print(cm_df)

Model saved to: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\models\unsw_cnn_baseline_model.h5
Results saved to: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\unsw_cnn_baseline_results.csv
Confusion matrix saved to: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results\unsw_cnn_confusion_matrix.csv

Results:
     Dataset         Model  Accuracy  Precision    Recall  F1-score  \
0  UNSW-NB15  Baseline CNN  0.922289   0.961005  0.923287  0.941769   

   Training Time Seconds  Inference Time Seconds  Model Size MB  
0              43.730621               14.660965       0.535698  

Confusion Matrix:
               Predicted Normal  Predicted Attack
Actual Normal             51529              4471
Actual Attack              9155            110186


In [10]:
start_time = time.time()

y_pred_prob = model.predict(X_test_cnn, verbose=0)
inference_time = time.time() - start_time

y_pred = (y_pred_prob >= 0.5).astype(int).ravel()

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("UNSW-NB15 CNN Baseline Results")
print(f"Accuracy:  {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall:    {recall:.6f}")
print(f"F1-score:  {f1:.6f}")
print(f"Inference time: {inference_time:.6f} seconds")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Attack"], zero_division=0))

UNSW-NB15 CNN Baseline Results
Accuracy:  0.922289
Precision: 0.961005
Recall:    0.923287
F1-score:  0.941769
Inference time: 14.660965 seconds

Classification Report:
              precision    recall  f1-score   support

      Normal       0.85      0.92      0.88     56000
      Attack       0.96      0.92      0.94    119341

    accuracy                           0.92    175341
   macro avg       0.91      0.92      0.91    175341
weighted avg       0.93      0.92      0.92    175341



In [9]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

start_time = time.time()

history = model.fit(
    X_train_cnn,
    y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch 1/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.9428 - loss: 0.1703 - val_accuracy: 0.1832 - val_loss: 1.6287
Epoch 2/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9594 - loss: 0.1130 - val_accuracy: 0.3859 - val_loss: 1.1984
Epoch 3/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9632 - loss: 0.0939 - val_accuracy: 0.5526 - val_loss: 1.0768
Epoch 4/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9667 - loss: 0.0829 - val_accuracy: 0.6214 - val_loss: 0.9168
Epoch 5/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9687 - loss: 0.0783 - val_accuracy: 0.6261 - val_loss: 0.9873
Epoch 6/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9701 - loss: 0.0739 - val_accuracy: 0.6723 - val_loss: 0.8271
Epoch 7/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9714 - loss: 0.0721 - val_accuracy: 0.6523 - val_loss: 0.8780
Epoch 8/20
515/515 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9721 - loss: 0.0695 - val_accuracy: 0.

In [8]:
input_shape = (X_train_cnn.shape[1], 1)

model = Sequential([
    Input(shape=input_shape),

    Conv1D(filters=32, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Conv1D(filters=64, kernel_size=3, activation="relu"),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    Flatten(),

    Dense(64, activation="relu"),
    Dropout(0.4),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 40, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 20, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 18, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 9, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 9, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,329 (169.25 KB)

 Trainable params: 43,329 (169.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print("CNN training shape:", X_train_cnn.shape)
print("CNN testing shape:", X_test_cnn.shape)

CNN training shape: (82332, 42, 1)
CNN testing shape: (175341, 42, 1)


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, SCALER_PATH)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)
print("Scaler saved to:", SCALER_PATH)

Scaled training shape: (82332, 42)
Scaled testing shape: (175341, 42)
Scaler saved to: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\models\unsw_scaler.pkl


In [5]:
TARGET_COLUMN = "label"
DROP_COLUMNS = ["label", "attack_cat"]

X_train = train_df.drop(columns=DROP_COLUMNS)
y_train = train_df[TARGET_COLUMN]

X_test = test_df.drop(columns=DROP_COLUMNS)
y_test = test_df[TARGET_COLUMN]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

X_train shape: (82332, 42)
y_train shape: (82332,)
X_test shape: (175341, 42)
y_test shape: (175341,)

Training target distribution:
label
1    45332
0    37000
Name: count, dtype: int64

Testing target distribution:
label
1    119341
0     56000
Name: count, dtype: int64


In [4]:
print("Training columns:")
print(train_df.columns.tolist())

print("\nLast 10 columns:")
print(train_df.columns[-10:].tolist())

print("\nUnique values in possible target columns:")
for col in train_df.columns[-10:]:
    print(col, train_df[col].nunique(), train_df[col].unique()[:10])

Training columns:
['dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Last 10 columns:
['ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Unique values in possible target columns:
ct_dst_sport_ltm 33 [ 1  2  3  4 10  6  5  8 18 14]
ct_dst_src_ltm 57 [ 2  3  1 14 17  7 11  9  4 18]
is_ftp_login 3 [0 1 2]
ct_ftp_cmd 3 [0 1 2]
ct_flw_http_mthd 8 [ 0  1  9  4  2 16 12  6]
ct_src_ltm 50 [ 1  2  3 10  4  5  9 11 22  6]
ct_srv_dst 57 

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

display(train_df.head())

Training data shape: (82332, 44)
Testing data shape: (175341, 44)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,6,0
3,0.000006,119,0,5,2,0,900,0,166666.6608,254,...,1,3,0,0,0,2,3,0,6,0
4,0.000010,119,0,5,2,0,2126,0,100000.0025,254,...,1,3,0,0,0,2,3,0,6,0


In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "datasets" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "unsw_train_processed.csv"
TEST_PATH = DATA_DIR / "unsw_test_processed.csv"

SCALER_PATH = MODEL_DIR / "unsw_scaler.pkl"
MODEL_PATH = MODEL_DIR / "unsw_cnn_baseline_model.h5"
RESULTS_PATH = RESULTS_DIR / "unsw_cnn_baseline_results.csv"
CONFUSION_MATRIX_PATH = RESULTS_DIR / "unsw_cnn_confusion_matrix.csv"

print("Train file exists:", TRAIN_PATH.exists())
print("Test file exists:", TEST_PATH.exists())
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)

Train file exists: True
Test file exists: True
Model directory: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\models
Results directory: C:\Users\Admin\Desktop\Lightweight-IoT-IDS\results


In [1]:
import os
import time
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dropout, Flatten, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

TensorFlow version: 2.21.0
NumPy version: 2.4.6
